This code is designed to get a NIST csv file for a specie and parse it for e.g. SIESTA.

In [1]:
import pandas as pd
import numpy as np
import re

In [7]:
class NISTParser():
    def __init__(self, filepath: str):
        self.filepath = filepath


    def toPandas(self,sep: str, header: int, print_head: bool) -> None:
        
        df = pd.read_csv(self.filepath, sep=sep, header=header)
        print(f"Dataframe loaded from {self.filepath.split('/')[-1]} has: {df.shape[0]} rows and {df.shape[1]} columns")
        print(f"Columns names are: {list(df.columns.values)}")
        
        if print_head:
            print(df.head(5))
        self.dataframe = df

    def TXTtoPandas(self, header: int, print_head: bool) -> None:
        with open(self.filepath, 'r') as file:
            lines = file.readlines()
        
        data = []
        for line in lines:
            if line.strip() and not line.startswith('#'):
                data.append(re.split(r'\s+', line.strip()))
        
        df = pd.DataFrame(data[header:], columns=data[header-1])
        print(f"Dataframe loaded from {self.filepath.split('/')[-1]} has: {df.shape[0]} rows and {df.shape[1]} columns")
        print(f"Columns names are: {list(df.columns.values)}")
        
        if print_head:
            print(df.head(5))
        self.dataframe = df
    
    def selectColumns(self, columns: list) -> None:
        if not hasattr(self, 'dataframe'):
            raise ValueError("Dataframe not loaded. Please run toPandas() method first.")
        
        if not all(col in self.dataframe.columns for col in columns):
            missing_cols = [col for col in columns if col not in self.dataframe.columns]
            raise ValueError(f"The following columns are not in the dataframe: {missing_cols}")
        
        selected_df = self.dataframe[columns]
        self.dataframe = selected_df
        print(f"Selected columns: {columns}")


    def intensityFilter(self, threshold_up: float) -> None:
        if not hasattr(self, 'dataframe'):
            raise ValueError("Dataframe not loaded. Please run toPandas() method first.")
        
        if 'intens' not in self.dataframe.columns:
            raise ValueError("Column 'intens' not found in dataframe.")
        
        self.dataframe["intens"] = (self.dataframe["intens"].astype(str).str.replace(r"[^0-9]", "", regex=True).replace("", np.nan).astype(float))



        filtered_df = self.dataframe[self.dataframe['intens'] >= np.quantile(self.dataframe['intens'], threshold_up)]
        self.dataframe = filtered_df
        print(f"Filtered dataframe to keep intensities >= {threshold_up}. New shape: {self.dataframe.shape}")
        
    
    def exportDataframeNumpy(self, output_filepath: str) -> None:
        if not hasattr(self, 'dataframe'):
            raise ValueError("Dataframe not loaded. Please run toPandas() method first.")
        
        exported_array = self.dataframe.to_numpy()
        np.save(output_filepath, exported_array)
        print(f"Dataframe exported to numpy array at {output_filepath}")

        

In [ ]:
# test.head(20)


# # Neon = NISTParser("./NIST_Atomic-Specie/NeI350-875nmtab.csv")
# Thorium = NISTParser("./NIST_Atomic-Specie/ThI350-875nmtab.csv")
# Argon = NISTParser("./NIST_Atomic-Specie/ArI350-875nmtab.csv")
# Argon.toPandas(sep='\t', header=0, print_head=False)
# Thorium.toPandas(sep='\t', header=0, print_head=False)
# Thorium.intensityFilter(threshold_up=0.90)
# # Argon.selectColumns(['intens'])
# Argon.intensityFilter(threshold_up=0.90)
# Argon.selectColumns(['obs_wl_air(nm)', 'intens'])

# Argon.selectColumns(['obs_wl_air(nm)'])
# print(Argon.dataframe.head(20))
# Argon.exportDataframeNumpy("./NIST_Atomic-Specie/Argon0.90.npy")
# # Thorium.selectColumns(['obs_wl_air(nm)'])
# # Thorium.selectColumns(['obs_wl_air(nm)', 'intens'])
# # Thorium.dataframe.head(20)
# # Thorium.exportDataframeNumpy("./NIST_Atomic-Specie/Thorium0.90.npy")

Dataframe loaded from ArI350-875nmtab.csv has: 311 rows and 17 columns
Columns names are: ['obs_wl_air(nm)', 'ritz_wl_air(nm)', 'intens', 'Aki(s^-1)', 'Acc', 'Ei(cm-1)', 'Ek(cm-1)', 'conf_i', 'term_i', 'J_i', 'conf_k', 'term_k', 'J_k', 'Type', 'tp_ref', 'line_ref', 'Unnamed: 16']
Dataframe loaded from ThI350-875nmtab.csv has: 8297 rows and 19 columns
Columns names are: ['obs_wl_air(nm)', 'unc_obs_wl', 'ritz_wl_air(nm)', 'unc_ritz_wl', 'intens', 'Aki(s^-1)', 'Acc', 'Ei(cm-1)', 'Ek(cm-1)', 'conf_i', 'term_i', 'J_i', 'conf_k', 'term_k', 'J_k', 'Type', 'tp_ref', 'line_ref', 'Unnamed: 18']
Filtered dataframe to keep intensities >= 0.9. New shape: (835, 19)
Filtered dataframe to keep intensities >= 0.9. New shape: (0, 17)
Selected columns: ['obs_wl_air(nm)', 'intens']
Selected columns: ['obs_wl_air(nm)']
Empty DataFrame
Columns: [obs_wl_air(nm)]
Index: []
Dataframe exported to numpy array at ./NIST_Atomic-Specie/Argon0.90.npy


In [8]:
ESO_THAR = NISTParser("./NIST_Atomic-Specie/ThAr_atlas_ESO.csv")
ESO_THAR.toPandas(sep='\t', header=0, print_head=True)
ESO_THAR.exportDataframeNumpy("./NIST_Atomic-Specie/ESO_THAR.npy")

Dataframe loaded from ThAr_atlas_ESO.csv has: 1845 rows and 1 columns
Columns names are: ['obs_wl_air(nm)']
   obs_wl_air(nm)
0        350.1867
1        350.3786
2        350.9779
3        351.1157
4        351.4388
Dataframe exported to numpy array at ./NIST_Atomic-Specie/ESO_THAR.npy


# PARSER FOR THAR ESO

One-time txt to csv, readable by NIST PARSER

In [ ]:
# # ESO_THAR = NISTParser("./NIST_Atomic-Specie/ThAr_atlas_ESO.txt")
# test = pd.read_csv(
#     "./NIST_Atomic-Specie/ThAr_atlas_ESO.txt",
#     sep= r"\s+",  # split on arbitrary spaces
#     header=None,            # no header in file
#     usecols=[1],            # keep only the right column
#     names=["obs_wl_air(nm)"]    # optional: give a column name
# )

# # test.head(20)
# select = test[(test["obs_wl_air(nm)"] >= 3500) & (test["obs_wl_air(nm)"] <= 8750)]
# select["obs_wl_air(nm)"] = select["obs_wl_air(nm)"] / 10
# # print(f"Dataframe loaded from {self.filepath.split('/')[-1]} has: {df.shape[0]} rows and {df.shape[1]} columns")
# # test.to_csv("./NIST_Atomic-Specie/ThAr_atlas_ESO.csv", index=False)
# select.head(20)
# select.to_csv("./NIST_Atomic-Specie/ThAr_atlas_ESO.csv", index=False, sep='\t')

